In [ ]:
# Shared paths: configure raw data once in config.local.toml at the repo root.
from pathlib import Path
import sys

_start = Path.cwd().resolve()
_project = next(
    (p for p in (_start, *_start.parents) if (p / "scripts" / "project_paths.py").is_file()),
    None,
)
if _project is None:
    raise RuntimeError("Start the notebook kernel in the repository or a subdirectory.")
_scripts = str(_project / "scripts")
if _scripts not in sys.path:
    sys.path.insert(0, _scripts)
from project_paths import PROJECT_ROOT, MIMIC_DATA_DIR, PROCESSED_DIR, REPORTS_DIR, mimic_csv


In [1]:
import pandas as pd
from pathlib import Path

processed_path = PROCESSED_DIR

train_data = pd.read_parquet(processed_path / "ml_train.parquet")
test_data = pd.read_parquet(processed_path / "ml_test.parquet")

X_train = train_data.drop(columns=["SUBJECT_ID", "HOSPITAL_EXPIRE_FLAG"])
y_train = train_data["HOSPITAL_EXPIRE_FLAG"]
subject_id_train = train_data["SUBJECT_ID"]

X_test = test_data.drop(columns=["SUBJECT_ID", "HOSPITAL_EXPIRE_FLAG"])
y_test = test_data["HOSPITAL_EXPIRE_FLAG"]

In [2]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

categorical_cols = [
    "gender",
    "admission_type",
    "admission_location"
]

numeric_cols = X_train.columns.difference(categorical_cols).tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", SimpleImputer(strategy="median"), numeric_cols),
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)

random_forest_model = RandomForestClassifier(
    random_state=42
)

random_forest_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", random_forest_model)
    ]
)

In [3]:
from sklearn.model_selection import StratifiedGroupKFold, cross_validate

setup = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

random_forest_results = cross_validate(
    random_forest_pipeline,
    X_train,
    y_train,
    cv=setup,
    groups=subject_id_train,
    scoring=["roc_auc", "precision", "recall", "f1"]
)

print("ROC-AUC:", random_forest_results["test_roc_auc"].mean())
print("Precision:", random_forest_results["test_precision"].mean())
print("Recall:", random_forest_results["test_recall"].mean())
print("F1:", random_forest_results["test_f1"].mean())

ROC-AUC: 0.8460232448206171
Precision: 0.7410106945574395
Recall: 0.15946708673190138
F1: 0.2623074259693342


In [4]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [5, 10, None],
    "model__min_samples_leaf": [1, 5],
    "model__max_features": ["sqrt", "log2"]
}

rf_grid_search = GridSearchCV(
    estimator=random_forest_pipeline,
    param_grid=param_grid,
    cv=setup,
    scoring="roc_auc",
    n_jobs=1
)

rf_grid_search.fit(
    X_train,
    y_train,
    groups=subject_id_train
)

print("Best parameters:", rf_grid_search.best_params_)
print("Best ROC-AUC:", rf_grid_search.best_score_)

Best parameters: {'model__max_depth': None, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 5, 'model__n_estimators': 200}
Best ROC-AUC: 0.8545357923689683


In [5]:
best_rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_leaf=5,
    max_features="sqrt",
    random_state=42
)

best_rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", best_rf_model)
    ]
)

best_rf_results = cross_validate(
    best_rf_pipeline,
    X_train,
    y_train,
    cv=setup,
    groups=subject_id_train,
    scoring=["roc_auc", "precision", "recall", "f1"]
)

print("ROC-AUC:", best_rf_results["test_roc_auc"].mean())
print("Precision:", best_rf_results["test_precision"].mean())
print("Recall:", best_rf_results["test_recall"].mean())
print("F1:", best_rf_results["test_f1"].mean())

ROC-AUC: 0.8545357923689683
Precision: 0.7982756167228218
Recall: 0.13497016379874244
F1: 0.23081699899143535


In [6]:
balanced_rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_leaf=5,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42
)

balanced_rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", balanced_rf_model)
    ]
)

balanced_rf_results = cross_validate(
    balanced_rf_pipeline,
    X_train,
    y_train,
    cv=setup,
    groups=subject_id_train,
    scoring=["roc_auc", "precision", "recall", "f1"]
)

print("ROC-AUC:", balanced_rf_results["test_roc_auc"].mean())
print("Precision:", balanced_rf_results["test_precision"].mean())
print("Recall:", balanced_rf_results["test_recall"].mean())
print("F1:", balanced_rf_results["test_f1"].mean())

ROC-AUC: 0.8616866543015703
Precision: 0.454538818175525
Recall: 0.5521200389806298
F1: 0.49846263766622867


## Random Forest

A Random Forest classifier was evaluated for ICU mortality prediction.

- Numerical features were processed using median imputation.
- Categorical features were transformed using `OneHotEncoder`.
- StandardScaler was not used because Random Forest is not sensitive to feature scaling.
- Model performance was evaluated using 5-fold `StratifiedGroupKFold`, keeping ICU stays from the same patient in the same fold.

### Baseline Random Forest

The default Random Forest produced:

- ROC-AUC: **0.846**
- Precision: **0.741**
- Recall: **0.159**
- F1: **0.262**

The model had strong precision and good discrimination, but recall was very low.

### Hyperparameter Tuning

`GridSearchCV` was used to test different Random Forest settings.

The best parameters were:

- `n_estimators = 200`
- `max_depth = None`
- `min_samples_leaf = 5`
- `max_features = "sqrt"`

With these parameters:

- ROC-AUC: **0.855**
- Precision: **0.798**
- Recall: **0.135**
- F1: **0.231**

Hyperparameter tuning slightly improved ROC-AUC and precision, but recall decreased further.

### Class Weight Balancing

Because mortality is the minority class, `class_weight="balanced"` was added to the tuned Random Forest.

The resulting performance was:

- ROC-AUC: **0.862**
- Precision: **0.455**
- Recall: **0.552**
- F1: **0.498**

Class weighting substantially improved recall while maintaining a reasonable precision level. It also produced the highest F1 score among the tested Random Forest configurations.

Threshold tuning was not applied because precision and recall were already relatively balanced, and changing the threshold would mainly trade one metric for the other rather than improve both simultaneously.

Overall, the tuned and class-balanced Random Forest provided the strongest and most balanced Random Forest performance for this dataset.